In [ ]:
# Gregory-Newton Forward Difference Table with Explicit Calculations

from fractions import Fraction
import math

# ---------------------------------------------------
# 1) Construir tabela numérica de diferenças
# ---------------------------------------------------
def build_diff_table_values(y_vals):
    """
    Retorna a tabela de diferenças como lista de colunas:
    diffs[0] = f(x_k)
    diffs[1] = Δ¹ f(x_k)
    diffs[2] = Δ² f(x_k)
    ...
    """
    n = len(y_vals)
    diffs = [y_vals[:]]  # cópia
    for k in range(1, n):
        prev = diffs[k-1]
        col = []
        for i in range(len(prev) - 1):
            col.append(prev[i+1] - prev[i])
        diffs.append(col)
    return diffs

# ---------------------------------------------------
# 2) Imprimir tabela "estilo livro", com as contas
# ---------------------------------------------------
def print_diff_table_explicit(x_vals, y_vals):
    n = len(x_vals)
    diffs = build_diff_table_values(y_vals)
    num_orders = len(diffs) - 1  # até Δ^{n-1}

    # table[r][c] = texto que vai naquela célula
    cols = 1 + len(diffs)  # x_k + Δ^0 + ... + Δ^{n-1}
    table = [[None] * cols for _ in range(n)]

    # primeira coluna: x_k
    for i in range(n):
        table[i][0] = str(x_vals[i])

    # segunda coluna: Δ^0 f(x_k) = f(x_k)
    for i in range(n):
        table[i][1] = str(y_vals[i])

    # demais colunas: diferenças com a conta explícita
    for k in range(1, len(diffs)):  # k = 1 .. n-1
        prev = diffs[k-1]
        col = diffs[k]
        for i in range(len(col)):
            a = prev[i+1]
            b = prev[i]
            # só coloca parênteses se for negativo, para ficar bonito
            if b < 0:
                expr = f"{a} - ({b}) = {col[i]}"
            else:
                expr = f"{a} - {b} = {col[i]}"
            table[i][k+1] = expr

    # nomes das colunas
    col_names = ["x_k", "Δ⁰ f(x_k)"]
    for k in range(1, len(diffs)):
        col_names.append(f"Δ^{k} f(x_k)")

    # calcular larguras
    col_widths = []
    for c in range(cols):
        max_len = len(col_names[c])
        for r in range(n):
            if table[r][c] is not None:
                max_len = max(max_len, len(table[r][c]))
        col_widths.append(max_len + 2)

    # cabeçalho
    header = ""
    for c in range(cols):
        header += col_names[c].ljust(col_widths[c])
    print(header)
    print("-" * sum(col_widths))

    # linhas
    for r in range(n):
        row = ""
        for c in range(cols):
            cell = table[r][c] if table[r][c] is not None else ""
            row += cell.ljust(col_widths[c])
        print(row)


# ---------------------------------------------------
# 3) Ferramentas para polinômios (lista de coeficientes)
# ---------------------------------------------------
def poly_mul_x_minus_a(p, a):
    """
    Multiplica um polinômio p(x) por (x - a).
    p = [c0, c1, c2, ...]  (c0 + c1 x + c2 x^2 + ...)
    """
    res = [Fraction(0)] * (len(p) + 1)
    for i, coef in enumerate(p):
        res[i+1] += coef        # vezes x
        res[i]   += -a * coef   # vezes (-a)
    return res

def poly_add(p, q):
    m = max(len(p), len(q))
    res = [Fraction(0)] * m
    for i in range(m):
        if i < len(p):
            res[i] += p[i]
        if i < len(q):
            res[i] += q[i]
    return res

def poly_scalar_mul(p, c):
    return [coef * c for coef in p]

def poly_to_string(p):
    """
    Converte [c0, c1, ..., cn] em string tipo "5*x^3 - 4*x + 30".
    """
    terms = []
    for power in range(len(p) - 1, -1, -1):
        coef = p[power]
        if coef == 0:
            continue

        sign = '+' if coef > 0 else '-'
        abs_coef = abs(coef)

        # coeficiente em texto
        if abs_coef.denominator == 1:
            c_str = str(abs_coef.numerator)
        else:
            c_str = f"{abs_coef.numerator}/{abs_coef.denominator}"

        if power == 0:
            term = c_str
        elif power == 1:
            term = "x" if abs_coef == 1 else f"{c_str}*x"
        else:
            term = f"x^{power}" if abs_coef == 1 else f"{c_str}*x^{power}"

        terms.append((sign, term))

    if not terms:
        return "0"

    first_sign, first_term = terms[0]
    expr = ""
    if first_sign == '-':
        expr += '-'
    expr += first_term

    for sign, term in terms[1:]:
        expr += f" {sign} {term}"

    return expr

# ---------------------------------------------------
# 4) Gregory–Newton passo a passo (forma progressiva)
# ---------------------------------------------------
def gregory_newton_step_by_step(x_vals, y_vals):
    x0 = x_vals[0]
    h = x_vals[1] - x0

    # tabela de diferenças (numérica) e depois em Fraction
    diffs_vals = build_diff_table_values(y_vals)
    diffs = [list(map(Fraction, col)) for col in diffs_vals]

    print("\n=== Tabela de diferenças (apenas valores) ===")
    for k, col in enumerate(diffs):
        print(f"Δ^{k}:", [float(c) for c in col])

    # P_0(x) = f(x0)
    P = [Fraction(diffs[0][0])]
    print("\n=== Passo 0 ===")
    print(f"P_0(x) = f(x0) = {poly_to_string(P)}")

    # base B_0(x) = 1
    B = [Fraction(1)]

    # termos de ordem 1..n-1
    for k in range(1, len(x_vals)):
        print(f"\n=== Passo {k}: termo de ordem {k} ===")
        delta_k0 = diffs[k][0]
        print(f"Δ^{k} f(x0) = {delta_k0}")

        # atualiza base B_k(x) = B_{k-1}(x) * (x - x_{k-1})
        B = poly_mul_x_minus_a(B, Fraction(x_vals[k-1]))
        print(f"B_{k}(x) = ∏(x - x_i), i=0..{k-1} = {poly_to_string(B)}")

        # coeficiente c_k
        c = delta_k0 / (Fraction(math.factorial(k)) * (Fraction(h) ** k))
        print(f"c_{k} = Δ^{k} f(x0) / ({k}! * h^{k})")
        print(f"    = {delta_k0} / ({math.factorial(k)} * {h}^{k}) = {c}")

        # termo T_k(x) = c_k * B_k(x)
        term = poly_scalar_mul(B, c)
        print(f"T_{k}(x) = c_{k} * B_{k}(x) = {poly_to_string(term)}")

        # P_k(x) = P_{k-1}(x) + T_k(x)
        P = poly_add(P, term)
        print(f"P_{k}(x) = P_{k-1}(x) + T_{k}(x) = {poly_to_string(P)}")

    print("\n=== Polinômio final ===")
    print("P(x) =", poly_to_string(P))
    return P

# ---------------------------------------------------
# 5) EXEMPLO DE USO COM O SEU PROBLEMA
# ---------------------------------------------------
if __name__ == "__main__":
    x_vals = [-2, -1, 0, 1, 2]
    y_vals = [-2, 29, 30, 31, 62]

    print("TABELA DE DIFERENÇAS (ESTILO LIVRO):\n")
    print_diff_table_explicit(x_vals, y_vals)

    print("\n\nCONSTRUÇÃO DO POLINÔMIO DE GREGORY–NEWTON:\n")
    gregory_newton_step_by_step(x_vals, y_vals)


TABELA DE DIFERENÇAS (ESTILO LIVRO):

x_k  Δ⁰ f(x_k)  Δ^1 f(x_k)      Δ^2 f(x_k)    Δ^3 f(x_k)      Δ^4 f(x_k)   
---------------------------------------------------------------------------
-2   -2         29 - (-2) = 31  1 - 31 = -30  0 - (-30) = 30  30 - 30 = 0  
-1   29         30 - 29 = 1     1 - 1 = 0     30 - 0 = 30                  
0    30         31 - 30 = 1     31 - 1 = 30                                
1    31         62 - 31 = 31                                               
2    62                                                                    


CONSTRUÇÃO DO POLINÔMIO DE GREGORY–NEWTON:


=== Tabela de diferenças (apenas valores) ===
Δ^0: [-2.0, 29.0, 30.0, 31.0, 62.0]
Δ^1: [31.0, 1.0, 1.0, 31.0]
Δ^2: [-30.0, 0.0, 30.0]
Δ^3: [30.0, 30.0]
Δ^4: [0.0]

=== Passo 0 ===
P_0(x) = f(x0) = -2

=== Passo 1: termo de ordem 1 ===
Δ^1 f(x0) = 31
B_1(x) = ∏(x - x_i), i=0..0 = x + 2
c_1 = Δ^1 f(x0) / (1! * h^1)
    = 31 / (1 * 1^1) = 31
T_1(x) = c_1 * B_1(x) = 31*x + 62
P_1(x) =

In [11]:
# Gregory-Newton Forward Difference Table with Explicit Calculations

from fractions import Fraction
import math

# ---------------------------------------------------
# 1) Construir tabela numérica de diferenças
# ---------------------------------------------------
def build_diff_table_values(y_vals):
    """
    Retorna a tabela de diferenças como lista de colunas:
    diffs[0] = f(x_k)
    diffs[1] = Δ¹ f(x_k)
    diffs[2] = Δ² f(x_k)
    ...
    """
    n = len(y_vals)
    diffs = [y_vals[:]]  # cópia
    for k in range(1, n):
        prev = diffs[k-1]
        col = []
        for i in range(len(prev) - 1):
            col.append(prev[i+1] - prev[i])
        diffs.append(col)
    return diffs

# ---------------------------------------------------
# 2) Imprimir tabela "estilo livro", com as contas
# ---------------------------------------------------
def print_diff_table_explicit(x_vals, y_vals):
    n = len(x_vals)
    diffs = build_diff_table_values(y_vals)
    num_orders = len(diffs) - 1  # até Δ^{n-1}

    # table[r][c] = texto que vai naquela célula
    cols = 1 + len(diffs)  # x_k + Δ^0 + ... + Δ^{n-1}
    table = [[None] * cols for _ in range(n)]

    # primeira coluna: x_k
    for i in range(n):
        table[i][0] = str(x_vals[i])

    # segunda coluna: Δ^0 f(x_k) = f(x_k)
    for i in range(n):
        table[i][1] = str(y_vals[i])

    # demais colunas: diferenças com a conta explícita
    for k in range(1, len(diffs)):  # k = 1 .. n-1
        prev = diffs[k-1]
        col = diffs[k]
        for i in range(len(col)):
            a = prev[i+1]
            b = prev[i]
            # só coloca parênteses se for negativo, para ficar bonito
            if b < 0:
                expr = f"{a} - ({b}) = {col[i]}"
            else:
                expr = f"{a} - {b} = {col[i]}"
            table[i][k+1] = expr

    # nomes das colunas
    col_names = ["x_k", "Δ⁰ f(x_k)"]
    for k in range(1, len(diffs)):
        col_names.append(f"Δ^{k} f(x_k)")

    # calcular larguras
    col_widths = []
    for c in range(cols):
        max_len = len(col_names[c])
        for r in range(n):
            if table[r][c] is not None:
                max_len = max(max_len, len(table[r][c]))
        col_widths.append(max_len + 2)

    # cabeçalho
    header = ""
    for c in range(cols):
        header += col_names[c].ljust(col_widths[c])
    print(header)
    print("-" * sum(col_widths))

    # linhas
    for r in range(n):
        row = ""
        for c in range(cols):
            cell = table[r][c] if table[r][c] is not None else ""
            row += cell.ljust(col_widths[c])
        print(row)


# ---------------------------------------------------
# 3) Ferramentas para polinômios (lista de coeficientes)
# ---------------------------------------------------
def poly_mul_x_minus_a(p, a):
    """
    Multiplica um polinômio p(x) por (x - a).
    p = [c0, c1, c2, ...]  (c0 + c1 x + c2 x^2 + ...)
    """
    res = [Fraction(0)] * (len(p) + 1)
    for i, coef in enumerate(p):
        res[i+1] += coef        # vezes x
        res[i]   += -a * coef   # vezes (-a)
    return res

def poly_add(p, q):
    m = max(len(p), len(q))
    res = [Fraction(0)] * m
    for i in range(m):
        if i < len(p):
            res[i] += p[i]
        if i < len(q):
            res[i] += q[i]
    return res

def poly_scalar_mul(p, c):
    return [coef * c for coef in p]

def poly_to_string(p):
    """
    Converte [c0, c1, ..., cn] em string tipo "5*x^3 - 4*x + 30".
    """
    terms = []
    for power in range(len(p) - 1, -1, -1):
        coef = p[power]
        if coef == 0:
            continue

        sign = '+' if coef > 0 else '-'
        abs_coef = abs(coef)

        # coeficiente em texto
        if abs_coef.denominator == 1:
            c_str = str(abs_coef.numerator)
        else:
            c_str = f"{abs_coef.numerator}/{abs_coef.denominator}"

        if power == 0:
            term = c_str
        elif power == 1:
            term = "x" if abs_coef == 1 else f"{c_str}*x"
        else:
            term = f"x^{power}" if abs_coef == 1 else f"{c_str}*x^{power}"

        terms.append((sign, term))

    if not terms:
        return "0"

    first_sign, first_term = terms[0]
    expr = ""
    if first_sign == '-':
        expr += '-'
    expr += first_term

    for sign, term in terms[1:]:
        expr += f" {sign} {term}"

    return expr

# ---------------------------------------------------
# 4) Gregory–Newton passo a passo (forma progressiva)
# ---------------------------------------------------
def gregory_newton_step_by_step(x_vals, y_vals):
    x0 = x_vals[0]
    h = x_vals[1] - x0

    # tabela de diferenças (numérica) e depois em Fraction
    diffs_vals = build_diff_table_values(y_vals)
    diffs = [list(map(Fraction, col)) for col in diffs_vals]

    print("\n=== Tabela de diferenças (apenas valores) ===")
    for k, col in enumerate(diffs):
        print(f"Δ^{k}:", [float(c) for c in col])

    # P_0(x) = f(x0)
    P = [Fraction(diffs[0][0])]
    print("\n=== Passo 0 ===")
    print(f"P_0(x) = f(x0) = {poly_to_string(P)}")

    # base B_0(x) = 1
    B = [Fraction(1)]

    # termos de ordem 1..n-1
    for k in range(1, len(x_vals)):
        print(f"\n=== Passo {k}: termo de ordem {k} ===")
        delta_k0 = diffs[k][0]
        print(f"Δ^{k} f(x0) = {delta_k0}")

        # atualiza base B_k(x) = B_{k-1}(x) * (x - x_{k-1})
        B = poly_mul_x_minus_a(B, Fraction(x_vals[k-1]))
        print(f"B_{k}(x) = ∏(x - x_i), i=0..{k-1} = {poly_to_string(B)}")

        # coeficiente c_k
        c = delta_k0 / (Fraction(math.factorial(k)) * (Fraction(h) ** k))
        print(f"c_{k} = Δ^{k} f(x0) / ({k}! * h^{k})")
        print(f"    = {delta_k0} / ({math.factorial(k)} * {h}^{k}) = {c}")

        # termo T_k(x) = c_k * B_k(x)
        term = poly_scalar_mul(B, c)
        print(f"T_{k}(x) = c_{k} * B_{k}(x) = {poly_to_string(term)}")

        # P_k(x) = P_{k-1}(x) + T_k(x)
        P = poly_add(P, term)
        print(f"P_{k}(x) = P_{k-1}(x) + T_{k}(x) = {poly_to_string(P)}")

    print("\n=== Polinômio final ===")
    print("P(x) =", poly_to_string(P))
    return P

# ---------------------------------------------------
# 5) EXEMPLO DE USO COM O SEU PROBLEMA
# ---------------------------------------------------
if __name__ == "__main__":
    x_vals = [-1, 0, 1, 2]
    y_vals = [3, 1, -1, 0]

    print("TABELA DE DIFERENÇAS (ESTILO LIVRO):\n")
    print_diff_table_explicit(x_vals, y_vals)

    print("\n\nCONSTRUÇÃO DO POLINÔMIO DE GREGORY–NEWTON:\n")
    gregory_newton_step_by_step(x_vals, y_vals)


TABELA DE DIFERENÇAS (ESTILO LIVRO):

x_k  Δ⁰ f(x_k)  Δ^1 f(x_k)    Δ^2 f(x_k)     Δ^3 f(x_k)  
---------------------------------------------------------
-1   3          1 - 3 = -2    -2 - (-2) = 0  3 - 0 = 3   
0    1          -1 - 1 = -2   1 - (-2) = 3               
1    -1         0 - (-1) = 1                             
2    0                                                   


CONSTRUÇÃO DO POLINÔMIO DE GREGORY–NEWTON:


=== Tabela de diferenças (apenas valores) ===
Δ^0: [3.0, 1.0, -1.0, 0.0]
Δ^1: [-2.0, -2.0, 1.0]
Δ^2: [0.0, 3.0]
Δ^3: [3.0]

=== Passo 0 ===
P_0(x) = f(x0) = 3

=== Passo 1: termo de ordem 1 ===
Δ^1 f(x0) = -2
B_1(x) = ∏(x - x_i), i=0..0 = x + 1
c_1 = Δ^1 f(x0) / (1! * h^1)
    = -2 / (1 * 1^1) = -2
T_1(x) = c_1 * B_1(x) = -2*x - 2
P_1(x) = P_0(x) + T_1(x) = -2*x + 1

=== Passo 2: termo de ordem 2 ===
Δ^2 f(x0) = 0
B_2(x) = ∏(x - x_i), i=0..1 = x^2 + x
c_2 = Δ^2 f(x0) / (2! * h^2)
    = 0 / (2 * 1^2) = 0
T_2(x) = c_2 * B_2(x) = 0
P_2(x) = P_1(x) + T_2(x) = -2*x 

In [ ]:
from fractions import Fraction

def eval_poly(poly, x):
    """
    Avalia o polinômio P(x) = a0 + a1 x + a2 x^2 + ...
    onde poly = [a0, a1, a2, ...]
    """
    x = Fraction(x)
    value = Fraction(0)
    pot = Fraction(1)  # x^0
    for coef in poly:
        value += coef * pot
        pot *= x
    return value


def verificar_polinomio(xs, ys, poly, usar_float=False, tol=1e-9):
    """
    Verifica se P(xi) = yi para todos os pontos.
    - xs: lista de x_i
    - ys: lista de f(x_i)
    - poly: lista de coeficientes (Fractions ou floats)
    - usar_float: se True, compara com tolerância (útil se coeficientes forem float)
    - tol: tolerância para comparação em ponto flutuante
    """
    print("Verificando se o polinômio interpola os pontos:")
    tudo_ok = True

    for xi, yi in zip(xs, ys):
        valor = eval_poly(poly, xi)

        if usar_float:
            # comparação com tolerância (caso esteja usando floats)
            dif = float(valor) - float(yi)
            ok = abs(dif) <= tol
        else:
            # comparação exata (Fraction)
            ok = (valor == yi)

        status = "OK ✅" if ok else "ERRO ❌"
        print(f"x = {xi:>4} : P(x) = {valor} ; f(x) = {yi}  ->  {status}")

        if not ok:
            tudo_ok = False

    if tudo_ok:
        print("\nResultado: o polinômio está CORRETO para todos os pontos. 🎉")
    else:
        print("\nResultado: o polinômio NÃO interpola todos os pontos. ⚠️")

    return tudo_ok

# Operações básicas com polinômios (lista de coeficientes)
# p = [a0, a1, a2, ...]  =>  a0 + a1 x + a2 x^2 + ...

def poly_add(p, q):
    m = max(len(p), len(q))
    r = [Fraction(0)] * m
    for i in range(m):
        if i < len(p):
            r[i] += p[i]
        if i < len(q):
            r[i] += q[i]
    return r

def poly_mul(p, q):
    r = [Fraction(0)] * (len(p) + len(q) - 1)
    for i, ai in enumerate(p):
        for j, bj in enumerate(q):
            r[i + j] += ai * bj
    return r

def poly_to_str(poly, var='x'):
    """Transforma [a0, a1, a2] em string tipo '1 - 7/3x + 2/3x^2'."""
    terms = []
    for i, coef in reversed(list(enumerate(poly))):
        if coef == 0:
            continue
        sign = "-" if coef < 0 else "+"
        abs_coef = -coef if coef < 0 else coef

        if abs_coef == 1 and i != 0:
            coef_str = ""
        else:
            coef_str = str(abs_coef)

        if i == 0:
            term = coef_str
        elif i == 1:
            term = f"{coef_str}{var}" if coef_str else var
        else:
            term = f"{coef_str}{var}^{i}" if coef_str else f"{var}^{i}"

        terms.append((sign, term))

    if not terms:
        return "0"

    first_sign, first_term = terms[0]
    s = ("-" if first_sign == "-" else "") + first_term
    for sign, term in terms[1:]:
        s += f" {sign} {term}"
    return s


def newton_step_by_step(xs, ys, var='x'):
    n = len(xs)

    # ===== 1) Tabela de diferenças divididas =====
    table = [[Fraction(0) for _ in range(n)] for _ in range(n)]
    for i in range(n):
        table[i][0] = Fraction(ys[i])

    print("Cálculo das diferenças divididas:")
    for ordem in range(1, n):
        print(f"\nOrdem {ordem}:")
        for i in range(n - ordem):
            num = table[i + 1][ordem - 1] - table[i][ordem - 1]
            den = Fraction(xs[i + ordem] - xs[i])
            table[i][ordem] = num / den
            print(
                f"f[x{i},...,x{i+ordem}] = "
                f"({table[i + 1][ordem - 1]} - {table[i][ordem - 1]}) "
                f"/ ({xs[i + ordem]} - {xs[i]}) = {table[i][ordem]}"
            )

    # Imprime tabela no estilo do slide
    print("\nTabela de diferenças divididas:")
    header = ["x"] + [f"ordem {k}" for k in range(n)]
    print(" | ".join(f"{h:^10}" for h in header))
    print("-" * (13 * len(header)))
    for i in range(n):
        row = [f"{xs[i]:^10}"]
        for k in range(n - i):
            row.append(f"{str(table[i][k]):^10}")
        print(" | ".join(row))

    # ===== 2) Forma de Newton simbólica =====
    print("\nForma geral de Newton (com os valores calculados):")
    partes = []
    for k in range(n):
        coef = table[0][k]
        if k == 0:
            partes.append(f"f[x0] = {coef}")
        else:
            fatores = "".join([f"({var} - {xs[j]})" for j in range(k)])
            partes.append(f"{fatores} * f[x0,...,x{k}] = {fatores} * ({coef})")
    print("P(x) = " + " + ".join(partes))

    # ===== 3) Construção P0, P1, P2,... passo a passo =====
    print("\nConstrução passo a passo do polinômio:")
    poly = [Fraction(0)]

    for k in range(n):
        coef = table[0][k]

        # base_k(x) = (x - x0)(x - x1)...(x - x_{k-1})
        if k == 0:
            base = [Fraction(1)]
            termo_poly = [coef]
            termo_str = str(coef)
        else:
            base = [Fraction(1)]
            for j in range(k):
                # (x - x_j)   =>  [-x_j, 1]
                base = poly_mul(base, [Fraction(-xs[j]), Fraction(1)])
            termo_poly = [coef * c for c in base]
            fatores = " * ".join([f"({var} - {xs[j]})" for j in range(k)])
            termo_str = f"{fatores} * ({coef})"

        print(f"\nTermo {k}:")
        if k == 0:
            print(f"T_{k}(x) = f[x0] = {termo_str}")
        else:
            print(f"Base_{k}(x) = " + " * ".join([f"({var} - {xs[j]})" for j in range(k)]))
            print(f"T_{k}(x) = Base_{k}(x) * f[x0,...,x{k}] = {termo_str}")

        # soma P_{k-1}(x) + T_k(x)
        poly = poly_add(poly, termo_poly)
        print(f"P_{k}(x) = {poly_to_str(poly, var)}")

    print("\nPolinômio final expandido:")
    print("P(x) = " + poly_to_str(poly, var))

    return table, poly


# ===== Exemplo do enunciado =====
xs = [-1, 0, 2]
ys = [4, 1, -1]

tabela, coeficientes = newton_step_by_step(xs, ys)


Cálculo das diferenças divididas:

Ordem 1:
f[x0,...,x1] = (1 - 4) / (0 - -1) = -3
f[x1,...,x2] = (-1 - 1) / (2 - 0) = -1

Ordem 2:
f[x0,...,x2] = (-1 - -3) / (2 - -1) = 2/3

Tabela de diferenças divididas:
    x      |  ordem 0   |  ordem 1   |  ordem 2  
----------------------------------------------------
    -1     |     4      |     -3     |    2/3    
    0      |     1      |     -1    
    2      |     -1    

Forma geral de Newton (com os valores calculados):
P(x) = f[x0] = 4 + (x - -1) * f[x0,...,x1] = (x - -1) * (-3) + (x - -1)(x - 0) * f[x0,...,x2] = (x - -1)(x - 0) * (2/3)

Construção passo a passo do polinômio:

Termo 0:
T_0(x) = f[x0] = 4
P_0(x) = 4

Termo 1:
Base_1(x) = (x - -1)
T_1(x) = Base_1(x) * f[x0,...,x1] = (x - -1) * (-3)
P_1(x) = -3x + 1

Termo 2:
Base_2(x) = (x - -1) * (x - 0)
T_2(x) = Base_2(x) * f[x0,...,x2] = (x - -1) * (x - 0) * (2/3)
P_2(x) = 2/3x^2 - 7/3x + 1

Polinômio final expandido:
P(x) = 2/3x^2 - 7/3x + 1


In [14]:
# ===== Exemplo do enunciado =====
xs = [-1, 0, 3]
ys = [15, 8, -1]

tabela, coeficientes = newton_step_by_step(xs, ys)
verificar_polinomio(xs, ys, coeficientes)

Cálculo das diferenças divididas:

Ordem 1:
f[x0,...,x1] = (8 - 15) / (0 - -1) = -7
f[x1,...,x2] = (-1 - 8) / (3 - 0) = -3

Ordem 2:
f[x0,...,x2] = (-3 - -7) / (3 - -1) = 1

Tabela de diferenças divididas:
    x      |  ordem 0   |  ordem 1   |  ordem 2  
----------------------------------------------------
    -1     |     15     |     -7     |     1     
    0      |     8      |     -3    
    3      |     -1    

Forma geral de Newton (com os valores calculados):
P(x) = f[x0] = 15 + (x - -1) * f[x0,...,x1] = (x - -1) * (-7) + (x - -1)(x - 0) * f[x0,...,x2] = (x - -1)(x - 0) * (1)

Construção passo a passo do polinômio:

Termo 0:
T_0(x) = f[x0] = 15
P_0(x) = 15

Termo 1:
Base_1(x) = (x - -1)
T_1(x) = Base_1(x) * f[x0,...,x1] = (x - -1) * (-7)
P_1(x) = -7x + 8

Termo 2:
Base_2(x) = (x - -1) * (x - 0)
T_2(x) = Base_2(x) * f[x0,...,x2] = (x - -1) * (x - 0) * (1)
P_2(x) = x^2 - 6x + 8

Polinômio final expandido:
P(x) = x^2 - 6x + 8
Verificando se o polinômio interpola os pontos:
x =   -

True

In [18]:
from fractions import Fraction

def spline_linear(xs, ys, var='x'):
    """
    Gera a spline linear interpolante e imprime passo a passo
    como no slide, sem tentar usar 'x' dentro de Fraction.
    """
    n = len(xs)

    print("=== SPLINE LINEAR INTERPOLANTE ===\n")
    splines = []

    for i in range(1, n):
        x0 = Fraction(xs[i-1])
        x1 = Fraction(xs[i])
        f0 = Fraction(ys[i-1])
        f1 = Fraction(ys[i])

        print(f"Intervalo {i}: [{x0}, {x1}]")
        print(f"f(x0) = {f0},  f(x1) = {f1}\n")

        # Fórmula da spline em cada intervalo:
        # s_i(x) = f(x0)*(x1 - x)/(x1 - x0) + f(x1)*(x - x0)/(x1 - x0)
        den = (x1 - x0)

        print("s_i(x) = f(x0)*(x1 - x)/(x1 - x0) + f(x1)*(x - x0)/(x1 - x0)")
        print(f"       = {f0} * ({x1} - {var})/({den})  +  {f1} * ({var} - {x0})/({den})")

        # Agora fazemos a conta só com números para achar A e B:
        # s_i(x) = A*x + B
        A = (f1 - f0) / den
        B = f0 - A * x0

        print("\nExpandindo (conta numérica):")
        print("Sabemos que a equação da reta é s_i(x) = A*x + B")
        print(f"A = (f1 - f0)/(x1 - x0) = ({f1} - {f0})/({x1} - {x0}) = {A}")
        print(f"B = f0 - A*x0 = {f0} - ({A})*{x0} = {B}")
        print(f"\nLogo: s_i(x) = {A}*{var} + {B}")
        print("-------------------------------------------\n")

        splines.append((A, B, (x0, x1)))

    return splines


def avaliar_spline(splines, x):
    """Avalia a spline em um x específico."""
    x = Fraction(x)
    for A, B, (a, b) in splines:
        if a <= x <= b:
            return A*x + B
    return None


def verificar_spline(xs, ys, splines):
    print("\n=== Verificando a Spline ===")
    ok = True
    for xi, yi in zip(xs, ys):
        val = avaliar_spline(splines, xi)
        status = "OK ✅" if val == yi else "ERRO ❌"
        print(f"Ponto x={xi}: spline={val}, esperado={yi} → {status}")
        if val != yi:
            ok = False
    if ok:
        print("\nSpline está CORRETA! 🎉")
    else:
        print("\nSpline NÃO interpola corretamente. ⚠️")
    return ok


# =========================
# EXEMPLO DO SLIDE
# =========================

xs = [1, 2, 5, 7]
ys = [1, 2, 3, Fraction(5, 2)]  # 2.5 = 5/2

splines = spline_linear(xs, ys)
verificar_spline(xs, ys, splines)


=== SPLINE LINEAR INTERPOLANTE ===

Intervalo 1: [1, 2]
f(x0) = 1,  f(x1) = 2

s_i(x) = f(x0)*(x1 - x)/(x1 - x0) + f(x1)*(x - x0)/(x1 - x0)
       = 1 * (2 - x)/(1)  +  2 * (x - 1)/(1)

Expandindo (conta numérica):
Sabemos que a equação da reta é s_i(x) = A*x + B
A = (f1 - f0)/(x1 - x0) = (2 - 1)/(2 - 1) = 1
B = f0 - A*x0 = 1 - (1)*1 = 0

Logo: s_i(x) = 1*x + 0
-------------------------------------------

Intervalo 2: [2, 5]
f(x0) = 2,  f(x1) = 3

s_i(x) = f(x0)*(x1 - x)/(x1 - x0) + f(x1)*(x - x0)/(x1 - x0)
       = 2 * (5 - x)/(3)  +  3 * (x - 2)/(3)

Expandindo (conta numérica):
Sabemos que a equação da reta é s_i(x) = A*x + B
A = (f1 - f0)/(x1 - x0) = (3 - 2)/(5 - 2) = 1/3
B = f0 - A*x0 = 2 - (1/3)*2 = 4/3

Logo: s_i(x) = 1/3*x + 4/3
-------------------------------------------

Intervalo 3: [5, 7]
f(x0) = 3,  f(x1) = 5/2

s_i(x) = f(x0)*(x1 - x)/(x1 - x0) + f(x1)*(x - x0)/(x1 - x0)
       = 3 * (7 - x)/(2)  +  5/2 * (x - 5)/(2)

Expandindo (conta numérica):
Sabemos que a equação da 

True